# 第7章：建立聊天應用程式
## OpenAI API 快速入門

本筆記本改編自[Azure OpenAI 範例庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，其中包含可存取[Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本。

Python OpenAI API 同樣適用於 Azure OpenAI 模型，但需要做些微調整。更多關於差異的資訊，請參考此處：[如何使用 Python 在 OpenAI 與 Azure OpenAI 端點間切換](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概覽  
「大型語言模型是將文字映射到文字的函數。給定一個輸入的文字字串，大型語言模型嘗試預測接下來將出現的文字」(1)。此「快速入門」筆記本將向使用者介紹大型語言模型的高階概念、使用 AML 起步的核心套件需求、提示設計的簡介，以及多個不同使用案例的簡短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 分類文本](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個簡短的練習將為提交提示給 OpenAI 模型進行簡單任務「摘要」提供基本介紹。


<strong>步驟</strong>：  
1. 在你的 python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫並設定你為所創建的 OpenAI 服務所配置的典型 OpenAI 安全憑證  
3. 為你的任務選擇一個模型  
4. 為模型建立一個簡單的提示  
5. 將你的請求提交到模型 API！


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助程式庫及建立憑證實例


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 搵啱嘅模型  
好似 GPT-4o 同 GPT-4o mini 呢啲模型可以理解同產生自然語言，並且喺 OpenAI 平台上提供唔同嘅性能同速度，適合唔同嘅工作任務。


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，透過在大量文本中訓練以最小化此預測錯誤，模型最終學會了對這些預測有用的概念。例如，它們會學會像」(1):

* 如何拼寫
* 語法如何運作
* 如何換句話說
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有輸入大型語言模型的元素中，影響最大的是文字提示」(1)。

可以用幾種方式提示大型語言模型來產生輸出：

指令：告訴模型你想要什麼
補全：誘導模型完成你想要的開頭部分
示範：向模型展示你想要的內容，包括：
提示中的少量示例
精調訓練資料集內多達數百或數千個示例」



#### 建立提示有三個基本準則：

<strong>展示並說明</strong>。透過指令、示例或兩者結合，清楚表達你想要什麼。如果你希望模型將列表中的項目按字母順序排序，或對段落進行情感分類，就明確地示範這是你要的。

<strong>提供優質資料</strong>。如果你想建立分類器或讓模型遵循某個模式，務必確保有足夠的示例。一定要校對你的示例——模型通常夠聰明能看穿基本拼寫錯誤並給出回應，但它也可能以為這是故意的，從而影響回答。

**檢查你的設定。** 溫度和 top_p 設定決定模型生成回應的確定性。如果你要求一個只有一個正確答案的回應，那麼你會想把這些調低；如果你想要多樣化的回應，則可能想把它們調高。人們在這些設定上犯的最大錯誤是假設它們是「聰明度」或「創造力」的控制器。


來源：https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果會有什麼不同？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 摘要文本  
#### 挑戰  
透過在文本段落末尾加入 "tl;dr:" 來摘要文本。注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以嘗試使用比 tl;dr 更具描述性的提示來修改模型的行為並自訂你收到的摘要(3)。  

最近的研究顯示，先在大型文本語料庫上進行預訓練，接著對特定任務微調，在許多自然語言處理任務和基準測試中取得了顯著進展。儘管架構通常與任務無關，這種方法仍需要數千甚至數萬個特定任務的微調數據集。相比之下，人類通常僅憑少量範例或簡單說明就能執行新的語言任務——這是當前自然語言處理系統仍然較難做到的。本文展示了擴大語言模型規模大幅提升了任務無關的少量示範能力，有時甚至能與先前最先進的微調方法競爭。 



摘要  


# 多種用例練習  
1. 摘要文本  
2. 分類文本  
3. 生成新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文本  
#### 挑戰  
在推理時將項目分類到提供的類別中。在下面的例子中，我們在提示中同時提供了類別和要分類的文本(*playground_reference)。 

客戶查詢：您好，我的筆記型電腦鍵盤上的一個鍵最近壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從範例詞彙中創造產品名稱。在提示中我們包含了關於我們將要產生名稱的產品資訊。我們也提供了相似範例以展示我們期望接收的模式。我們同時將溫度值設得較高，以增加隨機性及更具創新性的回應。

產品描述：一台家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙能適合任何腳型的鞋子。
種子詞：可調節、合腳、全適合。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 如需更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
